In [ ]:
import pandas as pd
import json
import os
import math
import librosa
import time
import librosa
import librosa.display
import scipy as sp
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#file address
raw_dict = {
    'b_good': '/content/drive/MyDrive/bronze and SS audio_corrected/Bronze/B_good_bead/',
    'b_bad': '/content/drive/MyDrive/bronze and SS audio_corrected/Bronze/B_bad_bead/',
}

In [ ]:
import os
def read_audio():
    """Read raw files"""
    new_dict = {}
    bead_id = 0
    for k,v in raw_dict.items(): #key, value that calls from raw_dict
        _list = []
        _type = ['SS','B'][k.startswith('b_')] #make a colomn for B or SS
        label = [1,0][k.endswith('good')] #label 0 if good and 1 if bad
        for filename in os.listdir(v):
            new_dict[bead_id] = {'type': _type, 'path': v+filename, 'label': label}
            bead_id+=1
    return new_dict

In [ ]:
new_dd = read_audio()
df = pd.DataFrame.from_dict(new_dd,orient='index')
df['bead_id'] = df.index
df.head()

In [ ]:
""" FFT creation """
import json
import os
import math
import librosa
import time
import librosa
import librosa.display
# import scipy as sp
from scipy.fftpack import fft
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np

SAMPLE_RATE = 8000
num_segments=10
SAMPLES_PER_TRACK=47495 #total number of samples for each data
samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)  

def give_fft(file_path):
    """ (segmentid, signal-segment, magnitude)"""
    signal1, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)

    Factor=int(len(signal1)/SAMPLES_PER_TRACK)
    
    #downsampled signal making
    s=0 #starting index
    e= SAMPLES_PER_TRACK*Factor #ending index
    signal=signal1[s:e:Factor]
    print(signal.shape)
    matrix = []
#     print(help(fft))
    # devide the signals into segments and then calculate the FFT of all segments of audio file.
    for d in range(num_segments):

        # calculate start and finish sample for current segment 
        start = samples_per_segment * d
        # print(start)
        finish = start + samples_per_segment

        ft = fft(signal[start:finish])
        magnitude = np.absolute(ft)
        matrix.append((d, signal[start:finish],magnitude))
    return matrix

    

In [ ]:
""" Create signal segment dataset """
test = df.copy()
test = test.loc[test.index.repeat(10)].reset_index(drop=True)
df_segment = test.sort_values(by=['type','label'])
df_segment.head(11)

In [ ]:
seg_id = []
ffts = []
signals = []

for i in range(len(df)):
    for d,signal,magnitude in give_fft(df.iloc[i].path):
        ffts.append(magnitude)
        signals.append(signal)
        seg_id.append(seg_id)
print(len(ffts))
print(df_segment.shape)
df_segment['fft'] = ffts
df_segment['signal'] = signals


In [ ]:
broze_df = df_segment[df_segment.type == 'B'][['label','fft']] # dataframe which has 2 column (label and fft)
steel_df = df_segment[df_segment.type == 'SS'][['label','fft']]
# steel_df.groupby(by='label').count()
# (broze_df.fft.apply(len) == 4915).all()
broze_df[['label']].to_csv('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_B_label.csv',index=False, header= False)
steel_df[['label']].to_csv('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_SS_label.csv',index=False, header= False)
df_segment[['label']].to_csv('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_M_label.csv',index = False, header = False)
# broze_df[['fft']].to_csv('FFT_B.csv',index=False, header= False)

In [ ]:
def transform(ddf):
    nparray = []
    for d in ddf:
        temp = []
        for i in d:
            temp.append(i)
        nparray.append(np.array(temp,dtype=np.float32).flatten())
    return np.array(nparray)

In [ ]:
np.savetxt('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_B.csv',transform(broze_df[['fft']].to_numpy()),delimiter=',')
np.savetxt('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_SS.csv',transform(steel_df[['fft']].to_numpy()),delimiter=',')
np.savetxt('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_M.csv',transform(df_segment[['fft']].to_numpy()),delimiter=',')



2nd part:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import squareform,pdist
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib
from pandas import DataFrame
# from ree import *
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
mat=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_SS.csv',header=None)
mat_ndarray=mat.values[:,0:4749]
print(mat.shape)

dist_matrix = pdist(mat_ndarray)
sq_matrix = squareform(dist_matrix)


mat2=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_SS_label.csv',header=None)


# print(wfeed)

target = mat2.values#need to change when necessary
color = ['g', 'r']
# print(target)


x = StandardScaler().fit_transform(mat_ndarray)#normalization
# print(x.shape)

# def run_PCA():
from sklearn.decomposition import PCA

#2D PCA
pca = PCA(copy=True, iterated_power='auto', n_components=25, random_state=1,
	    svd_solver='auto', tol=0.0, whiten=False) #can also use ipca 12 ta component k 2 ta conponent a vag korse
x_pca=pca.fit_transform(x)
print(x_pca.shape)
 
# np.savetxt("/content/drive/MyDrive/Colab Notebooks/PCA+KNN/PCA_B.csv", x_pca, delimiter=",")


fig, ax = plt.subplots()
ax.scatter(x_pca[:, 0], x_pca[:, 1],c=target,
				edgecolor='none', alpha=0.5, cmap='RdYlGn')
z = x_pca[:,0]
y = x_pca[:,1]

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
# plt.title("PCA_bronze_down_vw")
# plt.savefig("bronze_down_vw")
plt.show()
print(pca.explained_variance_ratio_)
 
# run_PCA() 

3rd : PCA classification

In [ ]:
import time


csv1=np.genfromtxt ('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/PCA_M.csv', delimiter=",")

csv2=np.genfromtxt ('/content/drive/MyDrive/Colab Notebooks/PCA+KNN/FFT_M_label.csv', delimiter=",")

def pca():
  X = csv1[:,:6] #26:78
  # print(X.shape)
  # print(X.shape)
  

  Y = csv2 

  X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.25)#split traing and testing dataset randomly
  X_train, X_validation, Y_train, y_validation = train_test_split(X_train, Y_train, test_size=0.20)
  
  # print(Y_test.shape)
  scaler = StandardScaler()
  scaler.fit(X_train)
  t0=time.time()
  X_train = scaler.transform(X_train)
  X_validation = scaler.transform(X_validation)
  X_test=scaler.transform(X_test)

  # print(X_test.shape)
  clf=KNeighborsClassifier(3)
  clf.fit(X_train, Y_train)
  # score = clf.score(X_test, Y_test)
  # print(score)
  t1=round(time.time()-t0, 3)
  # print ("training time:", t1, "s")

  t2=time.time()
  predict_y_test = clf.predict(X_test)
  t3=round(time.time()-t2, 3)
  # print ("testing time:", t3, "s")
  predict_y_train = clf.predict(X_train)
  predict_y_validation = clf.predict(X_validation)
  test_A=accuracy_score(Y_test,predict_y_test)
  train_A=accuracy_score(Y_train,predict_y_train)
  valid_A=accuracy_score(y_validation,predict_y_validation)
  
  return test_A,train_A,valid_A,t1,t3

 

pca()

In [ ]:
import numpy 
import matplotlib.pyplot as plt
import time
# time=[]
valid=[]
train=[]
test=[]
Test_time=[]
Train_time=[]
for i in range(30):
  
  test_acc,train_acc,val_acc,train_t,test_t=pca()
  test.append(test_acc)
  train.append(train_acc)
  valid.append(val_acc)
  Train_time.append(train_t)
  Test_time.append(test_t)

# print(train)
import numpy
print(numpy.mean(train))
print(numpy.mean(valid))
# print(numpy.mean(train[-1]))
# print(numpy.mean(valid[-1]))
print(numpy.mean(test))
print(numpy.std(test)) 